# Retail Analytics Pipeline — RetailMax
## Proyecto Integral: Módulo 9 — Fundamentos de Big Data
### 5 Lecciones en un solo notebook

> **Requisitos:** `pip install pyspark==3.5.1` + Java 8/11/17
> **Datos:** Ejecutar `cd data/ && python3 generate_data.py` si no existen los CSV

---
# ════════════════════════════════════════════════════════════
# Lección 1
# ════════════════════════════════════════════════════════════

# Lección 1: Fundamentos de Big Data
## Proyecto: Retail Analytics Pipeline — RetailMax

**Módulo 9 — Fundamentos de Big Data**

---

### 1.1 Las 5V de Big Data y su impacto en RetailMax

Big Data se define por cinco dimensiones fundamentales que determinan la complejidad y los requerimientos tecnológicos de un proyecto de análisis masivo.

#### 1. Volumen
RetailMax genera **millones de transacciones diarias** en su plataforma e-commerce. Solo en nuestro dataset de prueba tenemos ~500K transacciones, ~300K eventos de navegación y ~100K reseñas. En producción, estos volúmenes se multiplican por órdenes de magnitud.

**Impacto:** Requiere almacenamiento distribuido (HDFS, S3) y procesamiento paralelo (Spark) en lugar de soluciones monolíticas tradicionales.

#### 2. Velocidad
Los eventos de navegación ocurren en tiempo real: cada clic, cada vista de producto y cada búsqueda generan registros que deben capturarse y procesarse con baja latencia para alimentar sistemas de recomendación.

**Impacto:** Necesidad de streaming (Spark Structured Streaming, Kafka) para procesar datos en tiempo real o near-real-time.

#### 3. Variedad
Los datos de RetailMax son heterogéneos:
- **Estructurados:** transacciones (CSV, bases relacionales) con esquemas definidos.
- **Semi-estructurados:** logs de navegación (JSON), datos de APIs.
- **No estructurados:** reseñas en texto libre, imágenes de productos.

**Impacto:** Se requieren herramientas capaces de manejar múltiples formatos (Spark con soporte para CSV, JSON, Parquet, texto).

#### 4. Veracidad
Las reseñas pueden contener spam o reseñas falsas. Los registros de navegación pueden incluir bots. Los datos de transacciones pueden tener duplicados por reintentos de pago.

**Impacto:** Se necesitan pipelines de limpieza, detección de anomalías y validación de calidad de datos.

#### 5. Valor
El objetivo final es convertir estos datos masivos en **información accionable**: segmentación de clientes para marketing, predicción de compras, optimización de inventario y personalización de la experiencia.

**Impacto:** Machine Learning escalable (MLlib) para extraer valor de los datos procesados.

### 1.2 Fuentes de Datos de RetailMax

| Fuente | Tipo | Formato | Volumen estimado | Descripción |
|--------|------|---------|------------------|-------------|
| `transactions.csv` | Estructurado | CSV | ~500K registros | Compras: usuario, producto, monto, método de pago, dispositivo |
| `users.csv` | Estructurado | CSV | ~50K registros | Perfil de usuario: edad, género, región, segmento, LTV |
| `products.csv` | Estructurado | CSV | ~1K registros | Catálogo: categoría, precio, stock, rating promedio |
| `navigation.csv` | Semi-estructurado | CSV | ~300K registros | Comportamiento de navegación: páginas, tiempo, referrer |
| `reviews.csv` | No estructurado (texto) | CSV | ~100K registros | Reseñas y calificaciones de productos |

### 1.3 Diagrama de Arquitectura del Proyecto

```
┌─────────────────────────────────────────────────────────────────────┐
│                    ARQUITECTURA RETAILMAX BIG DATA                  │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌──────────────┐                                                   │
│  │  FUENTES DE  │                                                   │
│  │    DATOS     │                                                   │
│  ├──────────────┤     ┌───────────────────────────────┐             │
│  │transactions  │────►│                               │             │
│  │users         │────►│    INGESTA & ALMACENAMIENTO   │             │
│  │products      │────►│    (HDFS / S3 / Local FS)     │             │
│  │navigation    │────►│    Formato: CSV → Parquet     │             │
│  │reviews       │────►│                               │             │
│  └──────────────┘     └───────────┬───────────────────┘             │
│                                   │                                  │
│                                   ▼                                  │
│                       ┌───────────────────────┐                     │
│                       │   APACHE SPARK        │                     │
│                       │   (Motor de proceso)  │                     │
│                       ├───────────────────────┤                     │
│                       │ SparkContext           │                     │
│                       │ SparkSession           │                     │
│                       │                       │                     │
│                       │  ┌─────┐  ┌────────┐  │                     │
│                       │  │ RDD │  │DataFrm │  │                     │
│                       │  └──┬──┘  └───┬────┘  │                     │
│                       │     │         │       │                     │
│                       │     ▼         ▼       │                     │
│                       │  Transformaciones     │                     │
│                       │  map, filter, groupBy  │                     │
│                       │  join, agg, SQL       │                     │
│                       └───────────┬───────────┘                     │
│                                   │                                  │
│                    ┌──────────────┼──────────────┐                   │
│                    ▼              ▼              ▼                   │
│            ┌────────────┐ ┌────────────┐ ┌────────────┐             │
│            │ Spark SQL  │ │  Parquet   │ │   MLlib    │             │
│            │ Consultas  │ │  Storage   │ │  Pipeline  │             │
│            │ Métricas   │ │            │ │ LogReg     │             │
│            │ KPIs       │ │            │ │ K-Means    │             │
│            └────────────┘ └────────────┘ └─────┬──────┘             │
│                                                │                     │
│                                                ▼                     │
│                                    ┌───────────────────┐            │
│                                    │  RESULTADOS       │            │
│                                    │  • Segmentación   │            │
│                                    │  • Predicciones   │            │
│                                    │  • Dashboards     │            │
│                                    │  • Insights Mktg  │            │
│                                    └───────────────────┘            │
└─────────────────────────────────────────────────────────────────────┘
```

### 1.4 Stack Tecnológico

| Componente | Tecnología | Justificación |
|------------|-----------|---------------|
| Motor de procesamiento | Apache Spark 3.x | Procesamiento distribuido in-memory, soporte para RDDs, DataFrames y SQL |
| ML escalable | Spark MLlib | Integración nativa con Spark, pipelines escalables |
| Almacenamiento optimizado | Parquet | Formato columnar, compresión eficiente, schema evolution |
| Lenguaje | PySpark (Python) | Ecosistema de DS más amplio, integración con pandas/matplotlib |
| Entorno | Jupyter + PySpark local | Desarrollo iterativo, documentación integrada |

### 1.5 Plan de Ejecución

| Lección | Objetivo | Entregable |
|---------|----------|------------|
| L1 | Fundamentos conceptuales | Informe (este notebook) |
| L2 | Configuración Spark | Notebook con SparkSession + carga RDD |
| L3 | RDDs, transformaciones, acciones | Notebook con manipulación RDD + DAG |
| L4 | Spark SQL y DataFrames | Notebook + Parquet generados |
| L5 | MLlib Pipeline | Pipeline completo + informe final PDF |

---
*Fin de Lección 1 — Base conceptual establecida para el desarrollo técnico.*

---
# ════════════════════════════════════════════════════════════
# Lección 2
# ════════════════════════════════════════════════════════════

# Lección 2: Apache Spark — Introducción y Configuración
## Proyecto: Retail Analytics Pipeline — RetailMax

---

### Objetivo
Configurar SparkContext y SparkSession, cargar datos iniciales en RDDs y validar la conectividad con los datasets.

### Requisitos previos
```bash
pip install pyspark==3.5.1 findspark
```

> **Nota de compatibilidad Java:** PySpark 3.5.1 requiere Java 8, 11 o 17. Verificar con `java -version`. Si tienes Java 21+, instala Java 17 con `sudo apt install openjdk-17-jdk` o configura `JAVA_HOME` para apuntar a una versión compatible.

In [ ]:
# ─── Instalación de PySpark (ejecutar si es necesario) ───
# !pip install pyspark==3.5.1 findspark

# Si usas findspark para localizar la instalación de Spark:
# import findspark
# findspark.init()

In [ ]:
# ─── Imports ───────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark import SparkContext, SparkConf
import os
import time

print("Imports cargados correctamente ✓")

### 2.1 Configuración de SparkContext y SparkSession

Spark utiliza dos puntos de entrada principales:
- **SparkContext (sc):** API de bajo nivel para RDDs.
- **SparkSession (spark):** API unificada de alto nivel (DataFrames, SQL, Streaming).

Desde Spark 2.0+, SparkSession es el punto de entrada recomendado e incluye SparkContext internamente.

In [ ]:
# ─── Configuración de SparkSession ──────────────────────────
# Configuramos el entorno local simulando un clúster distribuido
spark = SparkSession.builder     .appName("RetailMax_BigData_Pipeline")     .master("local[*]")     .config("spark.sql.warehouse.dir", "./spark-warehouse")     .config("spark.driver.memory", "4g")     .config("spark.sql.shuffle.partitions", "8")     .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")     .getOrCreate()

# Obtener SparkContext desde la sesión
sc = spark.sparkContext

# Configurar nivel de log para reducir verbosidad
sc.setLogLevel("WARN")

print("=" * 60)
print("  SPARK SESSION CONFIGURADA")
print("=" * 60)
print(f"  App Name:        {spark.sparkContext.appName}")
print(f"  Master:          {spark.sparkContext.master}")
print(f"  Spark Version:   {spark.version}")
print(f"  Python Version:  {sc.pythonVer}")
print(f"  Cores:           {sc.defaultParallelism}")
print(f"  UI:              http://localhost:4040")
print("=" * 60)

### 2.2 Carga de Datos Iniciales en RDDs

Los RDDs (Resilient Distributed Datasets) son la abstracción fundamental de Spark. Son colecciones distribuidas, inmutables y tolerantes a fallos.

> **Importante:** Ajusta la variable `DATA_DIR` a la ruta donde están los archivos CSV en tu sistema.

In [ ]:
# ─── Definir rutas a los datos ──────────────────────────────
# ⚠️ AJUSTAR ESTA RUTA según tu instalación local
DATA_DIR = "../data/"

# Verificar que los archivos existen
datasets = ['products.csv', 'users.csv', 'transactions.csv',
            'navigation.csv', 'reviews.csv']

print("Verificando datasets...")
for ds in datasets:
    path = os.path.join(DATA_DIR, ds)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✓ {ds:25s} → {size_mb:.1f} MB")
    else:
        print(f"  ✗ {ds} NO ENCONTRADO — Verifica DATA_DIR")

In [ ]:
# ─── Carga de datos como RDDs de texto ──────────────────────
start = time.time()

# Cargar cada archivo como RDD de líneas de texto
rdd_transactions_raw = sc.textFile(os.path.join(DATA_DIR, "transactions.csv"))
rdd_users_raw = sc.textFile(os.path.join(DATA_DIR, "users.csv"))
rdd_products_raw = sc.textFile(os.path.join(DATA_DIR, "products.csv"))
rdd_navigation_raw = sc.textFile(os.path.join(DATA_DIR, "navigation.csv"))
rdd_reviews_raw = sc.textFile(os.path.join(DATA_DIR, "reviews.csv"))

elapsed = time.time() - start
print(f"Carga de RDDs completada en {elapsed:.2f} seg")
print("(Nota: La carga es lazy — no se lee realmente hasta la primera acción)")

In [ ]:
# ─── Acciones básicas: count y take ─────────────────────────
print("\n📊 Conteo de registros por dataset (acción: count)")
print("=" * 50)

start = time.time()

datasets_rdd = {
    'transactions': rdd_transactions_raw,
    'users': rdd_users_raw,
    'products': rdd_products_raw,
    'navigation': rdd_navigation_raw,
    'reviews': rdd_reviews_raw,
}

for name, rdd in datasets_rdd.items():
    n = rdd.count()
    print(f"  {name:20s}: {n:>10,} líneas")

elapsed = time.time() - start
print(f"\nTiempo total de conteo: {elapsed:.2f} seg")

In [ ]:
# ─── Explorar primeras líneas (acción: take) ───────────────
print("🔍 Primeras 3 líneas de transactions.csv:")
print("-" * 80)
for line in rdd_transactions_raw.take(3):
    print(line)

print("\n🔍 Primeras 3 líneas de reviews.csv:")
print("-" * 80)
for line in rdd_reviews_raw.take(3):
    print(line)

In [ ]:
# ─── Explorar particiones ───────────────────────────────────
print("📦 Número de particiones por RDD:")
print("=" * 50)
for name, rdd in datasets_rdd.items():
    print(f"  {name:20s}: {rdd.getNumPartitions()} particiones")

print("\n💡 Nota: El número de particiones determina el paralelismo.")
print("   En modo local[*], Spark usa tantos threads como cores disponibles.")

### 2.3 Validación de Conectividad

Verificamos que Spark puede leer correctamente los datos y que no hay problemas de encoding o formato.

In [ ]:
# ─── Validación: parsear una línea de transacciones ────────
header_tx = rdd_transactions_raw.first()
campos = header_tx.split(',')
print(f"Header: {header_tx}")
print(f"Campos: {len(campos)}")
print(f"Nombres: {campos}")

# Verificar que las líneas de datos tienen el mismo número de campos
sample_lines = rdd_transactions_raw.take(5)[1:]  # Skip header
for i, line in enumerate(sample_lines):
    n_fields = len(line.split(','))
    status = "✓" if n_fields == len(campos) else "✗"
    print(f"  Línea {i+1}: {n_fields} campos {status}")

In [ ]:
# ─── Resumen de la lección ──────────────────────────────────
print("\n" + "=" * 60)
print("  RESUMEN — LECCIÓN 2")
print("=" * 60)
print("  ✓ SparkSession y SparkContext configurados")
print("  ✓ 5 datasets cargados como RDDs")
print("  ✓ Total registros: ~951,000")
print("  ✓ Volumen total datos: ~80 MB")
print("  ✓ Validación de formato y conectividad exitosa")
print("=" * 60)

# NO cerrar Spark aquí — se reutiliza en las siguientes lecciones
# spark.stop()  # Descomentar solo al final del proyecto

---
# ════════════════════════════════════════════════════════════
# Lección 3
# ════════════════════════════════════════════════════════════

# Lección 3: Elementos Básicos de Spark (RDD, Transformaciones y Acciones)
## Proyecto: Retail Analytics Pipeline — RetailMax

---

### Objetivo
Manipular grandes volúmenes de datos mediante RDDs, aplicando transformaciones y acciones, y documentando el linaje (DAG).

In [ ]:
# ─── Setup ──────────────────────────────────────────────────
from pyspark.sql import SparkSession
import os
import time

spark = SparkSession.builder     .appName("RetailMax_L3_RDDs")     .master("local[*]")     .config("spark.driver.memory", "4g")     .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

# ⚠️ AJUSTAR ESTA RUTA según tu instalación local
DATA_DIR = "../data/"

print("Spark listo ✓")

### 3.1 Creación de RDDs a partir de Transacciones

Vamos a crear RDDs parseados a partir de los archivos CSV crudos, separando header de datos.

In [ ]:
# ─── Cargar y parsear transacciones ─────────────────────────
rdd_tx_raw = sc.textFile(os.path.join(DATA_DIR, "transactions.csv"))

# Extraer header
header_tx = rdd_tx_raw.first()
print(f"Header: {header_tx}")

# Filtrar header y parsear campos
# Nota: transactions.csv no tiene comas dentro de los campos,
# por lo que split(',') es seguro aquí.
rdd_tx = rdd_tx_raw     .filter(lambda line: line != header_tx)     .map(lambda line: line.split(","))

print(f"\nRegistros de transacciones: {rdd_tx.count():,}")
print(f"Ejemplo de registro parseado:")
print(rdd_tx.take(1))

In [ ]:
# ─── Crear RDD parseado con diccionarios ────────────────────
# Esquema: transaction_id, user_id, product_id, category, quantity,
#          unit_price, discount_pct, payment_method, device, timestamp, total_amount

def parse_transaction(fields):
    """Parsea una línea de transacción a un diccionario."""
    return {
        'transaction_id': fields[0],
        'user_id': fields[1],
        'product_id': fields[2],
        'category': fields[3],
        'quantity': int(fields[4]),
        'unit_price': float(fields[5]),
        'discount_pct': float(fields[6]),
        'payment_method': fields[7],
        'device': fields[8],
        'timestamp': fields[9],
        'total_amount': float(fields[10]),
    }

rdd_tx_parsed = rdd_tx.map(parse_transaction)

# Persistir en memoria para reutilización
rdd_tx_parsed.cache()

# Forzar materialización del caché con una acción
count_tx = rdd_tx_parsed.count()

print(f"RDD parseado y cacheado ✓ ({count_tx:,} registros)")
print(f"\nEjemplo: {rdd_tx_parsed.take(1)}")

### 3.2 Pair RDDs

Los Pair RDDs son RDDs de tuplas `(clave, valor)`. Son fundamentales para operaciones de agrupación, join y reducción.

In [ ]:
# ─── Crear Pair RDDs ────────────────────────────────────────

# Pair RDD: (categoría, total_amount)
rdd_cat_amount = rdd_tx_parsed.map(
    lambda tx: (tx['category'], tx['total_amount'])
)

# Pair RDD: (user_id, total_amount)
rdd_user_amount = rdd_tx_parsed.map(
    lambda tx: (tx['user_id'], tx['total_amount'])
)

# Pair RDD: (payment_method, 1) para conteo
rdd_payment_count = rdd_tx_parsed.map(
    lambda tx: (tx['payment_method'], 1)
)

print("Pair RDDs creados ✓")
print(f"\nEjemplos de (categoría, monto):")
for item in rdd_cat_amount.take(3):
    print(f"  {item}")

### 3.3 Transformaciones

Las transformaciones son operaciones **lazy** que definen un nuevo RDD sin ejecutarse inmediatamente. Spark construye un DAG (Directed Acyclic Graph) de transformaciones.

#### Tipos de transformaciones:
- **Narrow (estrechas):** map, filter, flatMap — no requieren shuffle entre particiones.
- **Wide (amplias):** reduceByKey, groupByKey, sortBy, distinct — requieren redistribución de datos.

In [ ]:
# ─── TRANSFORMACIÓN: map ────────────────────────────────────
# Extraer solo el monto total de cada transacción
rdd_amounts = rdd_tx_parsed.map(lambda tx: tx['total_amount'])
print(f"map → rdd_amounts: primeros 5 = {rdd_amounts.take(5)}")

In [ ]:
# ─── TRANSFORMACIÓN: filter ─────────────────────────────────
# Filtrar transacciones de alto valor (> $500)
rdd_high_value = rdd_tx_parsed.filter(lambda tx: tx['total_amount'] > 500)
n_high = rdd_high_value.count()
n_total = rdd_tx_parsed.count()
print(f"filter → Transacciones > $500: {n_high:,} de {n_total:,} ({n_high/n_total*100:.1f}%)")

In [ ]:
# ─── TRANSFORMACIÓN: flatMap ────────────────────────────────
# Cargar reseñas y extraer todas las palabras individuales
rdd_reviews_raw = sc.textFile(os.path.join(DATA_DIR, "reviews.csv"))
header_rev = rdd_reviews_raw.first()

# Usamos split con maxsplit=5 para manejar correctamente el campo de texto
# Campos: review_id(0), user_id(1), product_id(2), rating(3), review_text(4), timestamp(5)
rdd_words = rdd_reviews_raw     .filter(lambda line: line != header_rev)     .map(lambda line: line.split(",", 5))     .filter(lambda f: len(f) >= 5)     .flatMap(lambda f: f[4].lower().strip().split())

print(f"flatMap → Total de palabras en reseñas: {rdd_words.count():,}")
print(f"Primeras 10 palabras: {rdd_words.take(10)}")

In [ ]:
# ─── TRANSFORMACIÓN: distinct ───────────────────────────────
# Categorías únicas
rdd_categories = rdd_tx_parsed.map(lambda tx: tx['category']).distinct()
categories = rdd_categories.collect()
print(f"distinct → {len(categories)} categorías únicas:")
for cat in sorted(categories):
    print(f"  • {cat}")

In [ ]:
# ─── TRANSFORMACIÓN: sortBy ─────────────────────────────────
# Top 10 transacciones por monto
rdd_top_tx = rdd_tx_parsed     .sortBy(lambda tx: tx['total_amount'], ascending=False)

print("sortBy → Top 10 transacciones por monto:")
print(f"{'ID':<20} {'Categoría':<15} {'Monto':>10}")
print("-" * 48)
for tx in rdd_top_tx.take(10):
    print(f"{tx['transaction_id']:<20} {tx['category']:<15} ${tx['total_amount']:>9,.2f}")

In [ ]:
# ─── TRANSFORMACIÓN: reduceByKey (Pair RDD) ────────────────
# Ventas totales por categoría
rdd_sales_by_cat = rdd_cat_amount.reduceByKey(lambda a, b: a + b)

print("reduceByKey → Ventas totales por categoría:")
print(f"{'Categoría':<15} {'Ventas Totales':>15}")
print("-" * 32)
for cat, total in rdd_sales_by_cat.sortBy(lambda x: -x[1]).collect():
    print(f"{cat:<15} ${total:>14,.2f}")

In [ ]:
# ─── TRANSFORMACIÓN: reduceByKey para conteo ───────────────
# Conteo de transacciones por método de pago
rdd_payment_totals = rdd_payment_count.reduceByKey(lambda a, b: a + b)

print("reduceByKey → Transacciones por método de pago:")
print(f"{'Método':<25} {'Conteo':>10}")
print("-" * 37)
for method, count in rdd_payment_totals.sortBy(lambda x: -x[1]).collect():
    print(f"{method:<25} {count:>10,}")

### 3.4 Acciones

Las acciones **desencadenan la ejecución** del DAG completo. A diferencia de las transformaciones, las acciones producen un resultado concreto o efecto secundario.

In [ ]:
# ─── ACCIONES: collect, count, sum, mean, stdev ────────────
print("═" * 60)
print("  ESTADÍSTICAS DE MONTOS DE TRANSACCIÓN")
print("═" * 60)

# count
total_tx = rdd_amounts.count()
print(f"  count()  → Total transacciones: {total_tx:,}")

# sum (usando reduce)
total_revenue = rdd_amounts.reduce(lambda a, b: a + b)
print(f"  sum()    → Revenue total: ${total_revenue:,.2f}")

# mean
mean_amount = rdd_amounts.mean()
print(f"  mean()   → Ticket promedio: ${mean_amount:,.2f}")

# stdev
stdev_amount = rdd_amounts.stdev()
print(f"  stdev()  → Desv. estándar: ${stdev_amount:,.2f}")

# min / max
min_amount = rdd_amounts.min()
max_amount = rdd_amounts.max()
print(f"  min()    → Mínimo: ${min_amount:,.2f}")
print(f"  max()    → Máximo: ${max_amount:,.2f}")

print("═" * 60)

In [ ]:
# ─── ACCIÓN: countByKey ─────────────────────────────────────
# Conteo rápido de transacciones por dispositivo
rdd_device = rdd_tx_parsed.map(lambda tx: (tx['device'], 1))
device_counts = rdd_device.countByKey()

print("countByKey → Transacciones por dispositivo:")
for device, count in sorted(device_counts.items(), key=lambda x: -x[1]):
    pct = count / total_tx * 100
    print(f"  {device:<10}: {count:>10,} ({pct:.1f}%)")

### 3.5 Documentación del Linaje (DAG)

Spark construye un **DAG (Directed Acyclic Graph)** que representa la secuencia de transformaciones. El DAG define el linaje de datos y permite:
- Recalcular particiones perdidas (tolerancia a fallos)
- Optimizar la ejecución
- Visualizar el plan de ejecución

In [ ]:
# ─── Documentar el linaje del RDD ──────────────────────────
# El método toDebugString() muestra el linaje completo
print("📊 LINAJE (DAG) del RDD de ventas por categoría:")
print("=" * 60)
debug_str = rdd_sales_by_cat.toDebugString()
# toDebugString() puede retornar bytes o str según la versión de PySpark
if isinstance(debug_str, bytes):
    debug_str = debug_str.decode('utf-8')
print(debug_str)
print("=" * 60)
print()
print("📝 Interpretación del DAG:")
print("  1. textFile → Lee el CSV como líneas de texto (HadoopRDD)")
print("  2. filter   → Elimina la línea de header")
print("  3. map      → Parsea campos CSV a diccionario")
print("  4. map      → Extrae (categoría, monto) como Pair RDD")
print("  5. reduceByKey → Suma montos por categoría (shuffle)")
print()
print("💡 El 'shuffle' en reduceByKey redistribuye datos entre particiones")
print("   para agrupar todas las claves iguales en la misma partición.")

In [ ]:
# ─── Optimización: cache/persist ────────────────────────────
from pyspark import StorageLevel

# Verificar estado de caché
print("Estado de caché:")
print(f"  rdd_tx_parsed está en caché: {rdd_tx_parsed.is_cached}")

# Demostrar persist con nivel de almacenamiento
rdd_sales_by_cat.persist(StorageLevel.MEMORY_AND_DISK)
print(f"  rdd_sales_by_cat persistido: {rdd_sales_by_cat.is_cached}")

# Forzar materialización
_ = rdd_sales_by_cat.count()

print()
print("💡 Niveles de persistencia disponibles:")
print("  MEMORY_ONLY      → Solo en memoria (default de cache())")
print("  MEMORY_AND_DISK  → Memoria + disco si no cabe")
print("  DISK_ONLY        → Solo en disco")
print("  MEMORY_ONLY_SER  → Serializado en memoria (más compacto)")

In [ ]:
# ─── Resumen de Lección 3 ───────────────────────────────────
print("\n" + "=" * 60)
print("  RESUMEN — LECCIÓN 3")
print("=" * 60)
print("  ✓ RDDs creados y parseados desde CSV")
print("  ✓ Pair RDDs: (categoría, monto), (usuario, monto)")
print("  ✓ Transformaciones: map, filter, flatMap, distinct, sortBy, reduceByKey")
print("  ✓ Acciones: count, collect, reduce (sum), mean, stdev, min, max")
print("  ✓ Linaje DAG documentado con toDebugString()")
print("  ✓ Optimización con cache() y persist()")
print("=" * 60)

# NO cerrar Spark — se reutiliza en L4
# spark.stop()

---
# ════════════════════════════════════════════════════════════
# Lección 4
# ════════════════════════════════════════════════════════════

# Lección 4: Procesamiento de Datos Estructurados (Spark SQL y DataFrames)
## Proyecto: Retail Analytics Pipeline — RetailMax

---

### Objetivo
Procesar y analizar datos estructurados usando DataFrames y Spark SQL, generando métricas de negocio y guardando resultados en formato Parquet.

In [ ]:
# ─── Setup ──────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os
import time

spark = SparkSession.builder     .appName("RetailMax_L4_SparkSQL")     .master("local[*]")     .config("spark.driver.memory", "4g")     .config("spark.sql.shuffle.partitions", "8")     .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

# ⚠️ AJUSTAR ESTAS RUTAS según tu instalación local
DATA_DIR = "../data/"
OUTPUT_DIR = "../output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Spark listo ✓")

### 4.1 De RDDs a DataFrames con Esquemas Explícitos

Los DataFrames organizan los datos en columnas nombradas con tipos definidos (similar a una tabla SQL o un pandas DataFrame), lo que permite al optimizador Catalyst generar planes de ejecución eficientes.

In [ ]:
# ─── Definir esquemas explícitos ────────────────────────────
# Los esquemas explícitos evitan la inferencia automática (más rápido y preciso)

schema_transactions = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("device", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("total_amount", DoubleType(), True),
])

schema_users = StructType([
    StructField("user_id", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("region", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("lifetime_value", DoubleType(), True),
])

schema_products = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock", IntegerType(), True),
    StructField("rating_avg", DoubleType(), True),
    StructField("n_reviews", IntegerType(), True),
])

schema_reviews = StructType([
    StructField("review_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("rating", IntegerType(), True),
    StructField("review_text", StringType(), True),
    StructField("timestamp", StringType(), True),
])

print("Esquemas definidos ✓")

In [ ]:
# ─── Cargar CSVs como DataFrames ────────────────────────────
start = time.time()

df_transactions = spark.read.csv(
    os.path.join(DATA_DIR, "transactions.csv"),
    header=True, schema=schema_transactions
)

df_users = spark.read.csv(
    os.path.join(DATA_DIR, "users.csv"),
    header=True, schema=schema_users
)

df_products = spark.read.csv(
    os.path.join(DATA_DIR, "products.csv"),
    header=True, schema=schema_products
)

df_reviews = spark.read.csv(
    os.path.join(DATA_DIR, "reviews.csv"),
    header=True, schema=schema_reviews
)

# Cache de los DataFrames principales para reutilización
df_transactions.cache()
df_users.cache()

elapsed = time.time() - start
print(f"DataFrames cargados en {elapsed:.2f} seg ✓")
print(f"\nConteos:")
print(f"  Transacciones: {df_transactions.count():,}")
print(f"  Usuarios:      {df_users.count():,}")
print(f"  Productos:     {df_products.count():,}")
print(f"  Reseñas:       {df_reviews.count():,}")

In [ ]:
# ─── Inspeccionar esquema y datos ───────────────────────────
print("📋 Esquema de transacciones:")
df_transactions.printSchema()
print("\nPrimeros registros:")
df_transactions.show(5, truncate=False)

### 4.2 Registro de Tablas Temporales para Spark SQL

In [ ]:
# ─── Registrar como tablas temporales ───────────────────────
df_transactions.createOrReplaceTempView("transactions")
df_users.createOrReplaceTempView("users")
df_products.createOrReplaceTempView("products")
df_reviews.createOrReplaceTempView("reviews")

print("Tablas registradas: transactions, users, products, reviews ✓")
print("\nTablas disponibles:")
spark.sql("SHOW TABLES").show()

### 4.3 Consultas SQL — Métricas de Negocio

In [ ]:
# ─── MÉTRICA 1: Ventas totales por categoría ───────────────
print("📊 VENTAS TOTALES POR CATEGORÍA")
print("=" * 55)

sql_ventas_cat = spark.sql("""
    SELECT
        category AS Categoria,
        COUNT(*) AS Num_Transacciones,
        ROUND(SUM(total_amount), 2) AS Ventas_Totales,
        ROUND(AVG(total_amount), 2) AS Ticket_Promedio,
        ROUND(AVG(discount_pct), 1) AS Descuento_Prom_Pct
    FROM transactions
    GROUP BY category
    ORDER BY Ventas_Totales DESC
""")

sql_ventas_cat.show(15, truncate=False)

In [ ]:
# ─── MÉTRICA 2: Top 20 productos más vendidos ──────────────
print("📊 TOP 20 PRODUCTOS MÁS VENDIDOS")
print("=" * 55)

sql_top_prod = spark.sql("""
    SELECT
        t.product_id,
        p.product_name,
        p.category,
        p.subcategory,
        COUNT(*) AS total_ventas,
        ROUND(SUM(t.total_amount), 2) AS revenue_total,
        ROUND(AVG(t.total_amount), 2) AS ticket_promedio
    FROM transactions t
    JOIN products p ON t.product_id = p.product_id
    GROUP BY t.product_id, p.product_name, p.category, p.subcategory
    ORDER BY revenue_total DESC
    LIMIT 20
""")

sql_top_prod.show(20, truncate=False)

In [ ]:
# ─── MÉTRICA 3: Análisis por segmento de usuario ──────────
print("📊 MÉTRICAS POR SEGMENTO DE USUARIO")
print("=" * 55)

sql_segmentos = spark.sql("""
    SELECT
        u.segment AS Segmento,
        COUNT(DISTINCT u.user_id) AS Usuarios,
        COUNT(t.transaction_id) AS Transacciones,
        ROUND(SUM(t.total_amount), 2) AS Revenue,
        ROUND(AVG(t.total_amount), 2) AS Ticket_Prom,
        ROUND(SUM(t.total_amount) / COUNT(DISTINCT u.user_id), 2) AS Revenue_Por_Usuario
    FROM users u
    LEFT JOIN transactions t ON u.user_id = t.user_id
    GROUP BY u.segment
    ORDER BY Revenue DESC
""")

sql_segmentos.show(truncate=False)

In [ ]:
# ─── MÉTRICA 4: Evolución mensual de ventas ────────────────
print("📊 EVOLUCIÓN MENSUAL DE VENTAS")
print("=" * 55)

sql_monthly = spark.sql("""
    SELECT
        SUBSTRING(timestamp, 1, 7) AS Mes,
        COUNT(*) AS Transacciones,
        ROUND(SUM(total_amount), 2) AS Revenue,
        COUNT(DISTINCT user_id) AS Usuarios_Unicos,
        ROUND(AVG(total_amount), 2) AS Ticket_Promedio
    FROM transactions
    GROUP BY SUBSTRING(timestamp, 1, 7)
    ORDER BY Mes
""")

sql_monthly.show(20, truncate=False)

In [ ]:
# ─── MÉTRICA 5: Análisis por dispositivo y método de pago ──
print("📊 REVENUE POR DISPOSITIVO × MÉTODO DE PAGO")
print("=" * 55)

sql_device_pay = spark.sql("""
    SELECT
        device AS Dispositivo,
        payment_method AS Metodo_Pago,
        COUNT(*) AS N_Transacciones,
        ROUND(SUM(total_amount), 2) AS Revenue,
        ROUND(AVG(total_amount), 2) AS Ticket_Prom
    FROM transactions
    GROUP BY device, payment_method
    ORDER BY device, Revenue DESC
""")

sql_device_pay.show(20, truncate=False)

In [ ]:
# ─── MÉTRICA 6: Rating promedio por categoría + revenue ────
print("📊 SATISFACCIÓN vs REVENUE POR CATEGORÍA")
print("=" * 55)

sql_satisfaction = spark.sql("""
    SELECT
        p.category AS Categoria,
        ROUND(AVG(r.rating), 2) AS Rating_Promedio,
        COUNT(r.review_id) AS Total_Resenas,
        ROUND(SUM(t.total_amount), 2) AS Revenue_Total
    FROM products p
    LEFT JOIN reviews r ON p.product_id = r.product_id
    LEFT JOIN transactions t ON p.product_id = t.product_id
    GROUP BY p.category
    ORDER BY Rating_Promedio DESC
""")

sql_satisfaction.show(truncate=False)

### 4.4 Análisis con DataFrame API

In [ ]:
# ─── Análisis de distribución regional ─────────────────────
print("📊 DISTRIBUCIÓN REGIONAL DE VENTAS")
print("=" * 55)

df_regional = df_transactions     .join(df_users, "user_id")     .groupBy("region")     .agg(
        F.count("transaction_id").alias("n_transacciones"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("ticket_prom"),
        F.countDistinct("user_id").alias("usuarios_unicos"),
    )     .orderBy(F.desc("revenue"))

df_regional.show(truncate=False)

In [ ]:
# ─── Window Functions: Ranking de productos por categoría ──
print("📊 TOP 3 PRODUCTOS POR CATEGORÍA (Window Function)")
print("=" * 55)

w = Window.partitionBy("category").orderBy(F.desc("revenue"))

df_prod_rank = df_transactions     .groupBy("product_id", "category")     .agg(F.round(F.sum("total_amount"), 2).alias("revenue"))     .withColumn("rank", F.rank().over(w))     .filter(F.col("rank") <= 3)     .orderBy("category", "rank")

df_prod_rank.show(30, truncate=False)

### 4.5 Guardar Resultados en Formato Parquet

Parquet es un formato **columnar** optimizado para consultas analíticas:
- Compresión eficiente (solo lee las columnas necesarias)
- Schema embebido en los metadatos
- Soportado nativamente por Spark, Hive, Presto, etc.

In [ ]:
# ─── Guardar DataFrames procesados en Parquet ──────────────
PARQUET_DIR = os.path.join(OUTPUT_DIR, "parquet")
os.makedirs(PARQUET_DIR, exist_ok=True)

# 1. Transacciones enriquecidas (join con usuarios)
df_tx_enriched = df_transactions     .join(df_users.select("user_id", "age", "gender", "region", "segment"), "user_id")     .withColumn("month", F.substring("timestamp", 1, 7))     .withColumn("year", F.substring("timestamp", 1, 4))

df_tx_enriched.write.mode("overwrite").parquet(
    os.path.join(PARQUET_DIR, "transactions_enriched")
)
print(f"✓ transactions_enriched.parquet guardado ({df_tx_enriched.count():,} registros)")

# 2. Ventas por categoría
sql_ventas_cat.write.mode("overwrite").parquet(
    os.path.join(PARQUET_DIR, "ventas_por_categoria")
)
print("✓ ventas_por_categoria.parquet guardado")

# 3. Evolución mensual
sql_monthly.write.mode("overwrite").parquet(
    os.path.join(PARQUET_DIR, "evolucion_mensual")
)
print("✓ evolucion_mensual.parquet guardado")

# 4. Métricas por segmento
sql_segmentos.write.mode("overwrite").parquet(
    os.path.join(PARQUET_DIR, "metricas_segmento")
)
print("✓ metricas_segmento.parquet guardado")

In [ ]:
# ─── Verificar archivos Parquet ─────────────────────────────
print("📂 Archivos Parquet generados:")
for item in sorted(os.listdir(PARQUET_DIR)):
    item_path = os.path.join(PARQUET_DIR, item)
    if os.path.isdir(item_path):
        size = sum(
            os.path.getsize(os.path.join(item_path, f))
            for f in os.listdir(item_path)
            if not f.startswith('.')
        )
        print(f"  📁 {item}/ → {size / (1024*1024):.2f} MB")

# Verificar lectura
print("\nVerificando lectura de Parquet:")
df_test = spark.read.parquet(os.path.join(PARQUET_DIR, "transactions_enriched"))
print(f"✓ Re-lectura exitosa: {df_test.count():,} registros")
df_test.printSchema()

In [ ]:
# ─── Resumen de Lección 4 ───────────────────────────────────
print("\n" + "=" * 60)
print("  RESUMEN — LECCIÓN 4")
print("=" * 60)
print("  ✓ DataFrames creados con esquemas explícitos (StructType)")
print("  ✓ Tablas temporales registradas para Spark SQL")
print("  ✓ 6 consultas SQL con métricas de negocio:")
print("    - Ventas por categoría")
print("    - Top 20 productos")
print("    - Análisis por segmento")
print("    - Evolución mensual")
print("    - Dispositivo × método de pago")
print("    - Satisfacción vs revenue")
print("  ✓ DataFrame API: análisis regional + window functions")
print("  ✓ Resultados guardados en formato Parquet")
print("=" * 60)

# NO cerrar Spark — se reutiliza en L5
# spark.stop()

---
# ════════════════════════════════════════════════════════════
# Lección 5
# ════════════════════════════════════════════════════════════

# Lección 5: Introducción a Machine Learning Escalable (Spark MLlib)
## Proyecto: Retail Analytics Pipeline — RetailMax

---

### Objetivo
Construir un pipeline de MLlib para clasificación supervisada (Regresión Logística) y segmentación no supervisada (K-Means) de usuarios.

In [ ]:
# ─── Setup ──────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# MLlib imports
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, OneHotEncoder,
    StandardScaler, MinMaxScaler
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)

import os
import time

spark = SparkSession.builder     .appName("RetailMax_L5_MLlib")     .master("local[*]")     .config("spark.driver.memory", "4g")     .config("spark.sql.shuffle.partitions", "8")     .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

# ⚠️ AJUSTAR ESTAS RUTAS
DATA_DIR = "../data/"
OUTPUT_DIR = "../output/"
PARQUET_DIR = os.path.join(OUTPUT_DIR, "parquet")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Spark + MLlib listos ✓")

### 5.1 Carga de DataFrames Procesados

Cargamos los datos enriquecidos de la Lección 4 (Parquet). Si no están disponibles, reconstruimos desde CSV.

In [ ]:
# ─── Cargar datos ──────────────────────────────────────────
parquet_path = os.path.join(PARQUET_DIR, "transactions_enriched")

if os.path.exists(parquet_path):
    df_enriched = spark.read.parquet(parquet_path)
    print(f"✓ Cargado desde Parquet: {df_enriched.count():,} registros")
else:
    # Fallback: reconstruir desde CSV
    print("⚠ Parquet no disponible, reconstruyendo desde CSV...")
    print("  (Ejecuta primero el notebook de Lección 4 para generar los Parquet)")

    schema_tx = StructType([
        StructField("transaction_id", StringType()),
        StructField("user_id", StringType()),
        StructField("product_id", StringType()),
        StructField("category", StringType()),
        StructField("quantity", IntegerType()),
        StructField("unit_price", DoubleType()),
        StructField("discount_pct", DoubleType()),
        StructField("payment_method", StringType()),
        StructField("device", StringType()),
        StructField("timestamp", StringType()),
        StructField("total_amount", DoubleType()),
    ])

    schema_users = StructType([
        StructField("user_id", StringType()),
        StructField("age", IntegerType()),
        StructField("gender", StringType()),
        StructField("region", StringType()),
        StructField("registration_date", StringType()),
        StructField("segment", StringType()),
        StructField("lifetime_value", DoubleType()),
    ])

    df_tx = spark.read.csv(os.path.join(DATA_DIR, "transactions.csv"), header=True, schema=schema_tx)
    df_users = spark.read.csv(os.path.join(DATA_DIR, "users.csv"), header=True, schema=schema_users)

    df_enriched = df_tx.join(
        df_users.select("user_id", "age", "gender", "region", "segment"),
        "user_id"
    ).withColumn("month", F.substring("timestamp", 1, 7))

    print(f"✓ Reconstruido: {df_enriched.count():,} registros")

df_enriched.cache()
df_enriched.printSchema()

### 5.2 Feature Engineering para ML

Construimos features a nivel de usuario, agregando métricas de comportamiento de compra.

In [ ]:
# ─── Crear features agregadas por usuario ──────────────────
df_user_features = df_enriched.groupBy("user_id").agg(
    # Métricas de compra
    F.count("transaction_id").alias("total_purchases"),
    F.round(F.sum("total_amount"), 2).alias("total_spent"),
    F.round(F.avg("total_amount"), 2).alias("avg_ticket"),
    F.round(F.stddev("total_amount"), 2).alias("std_ticket"),
    F.round(F.avg("quantity"), 2).alias("avg_quantity"),
    F.round(F.avg("discount_pct"), 2).alias("avg_discount"),

    # Diversidad de compra
    F.countDistinct("category").alias("n_categories"),
    F.countDistinct("product_id").alias("n_products"),
    F.countDistinct("payment_method").alias("n_payment_methods"),

    # Información demográfica (primera ocurrencia)
    F.first("age").alias("age"),
    F.first("gender").alias("gender"),
    F.first("region").alias("region"),
    F.first("segment").alias("segment"),

    # Dispositivo predominante
    F.first("device").alias("primary_device"),
)

# Llenar nulos de std_ticket (usuarios con 1 sola compra → stddev = null)
df_user_features = df_user_features.fillna({"std_ticket": 0.0})

print(f"Features por usuario: {df_user_features.count():,} registros")
print(f"Columnas: {len(df_user_features.columns)}")
df_user_features.show(5, truncate=False)

### 5.3 Modelo Supervisado: Regresión Logística

**Objetivo:** Clasificar usuarios en "alto valor" vs "bajo valor" basándose en su comportamiento de compra.

Se define "alto valor" como usuarios cuyo gasto total supera el percentil 75.

In [ ]:
# ─── Crear variable objetivo (label) ───────────────────────
# Calcular el percentil 75 de gasto total
p75 = df_user_features.approxQuantile("total_spent", [0.75], 0.01)[0]
print(f"Percentil 75 de gasto total: ${p75:,.2f}")

# Crear label binaria: 1 = alto valor, 0 = bajo valor
df_ml = df_user_features.withColumn(
    "label",
    F.when(F.col("total_spent") >= p75, 1.0).otherwise(0.0)
)

# Verificar distribución
print("\nDistribución de la variable objetivo:")
df_ml.groupBy("label").agg(
    F.count("*").alias("n_usuarios"),
    F.round(F.avg("total_spent"), 2).alias("avg_gasto")
).show()

In [ ]:
# ─── Pipeline de preprocesamiento ──────────────────────────
# Indexar variables categóricas
indexer_gender = StringIndexer(
    inputCol="gender", outputCol="gender_idx", handleInvalid="keep"
)
indexer_region = StringIndexer(
    inputCol="region", outputCol="region_idx", handleInvalid="keep"
)
indexer_device = StringIndexer(
    inputCol="primary_device", outputCol="device_idx", handleInvalid="keep"
)

# One-Hot Encoding (handleInvalid="keep" para manejar categorías no vistas en test)
encoder_gender = OneHotEncoder(
    inputCol="gender_idx", outputCol="gender_vec", handleInvalid="keep"
)
encoder_region = OneHotEncoder(
    inputCol="region_idx", outputCol="region_vec", handleInvalid="keep"
)
encoder_device = OneHotEncoder(
    inputCol="device_idx", outputCol="device_vec", handleInvalid="keep"
)

# Features numéricas a incluir en el modelo
numeric_cols = [
    "total_purchases", "avg_ticket", "std_ticket",
    "avg_quantity", "avg_discount", "n_categories",
    "n_products", "n_payment_methods", "age"
]

# VectorAssembler: combinar todas las features en un solo vector
assembler = VectorAssembler(
    inputCols=numeric_cols + ["gender_vec", "region_vec", "device_vec"],
    outputCol="features_raw",
    handleInvalid="skip"  # Saltar filas con NaN/null
)

# Escalar features
# NOTA: Usamos withMean=False porque OneHotEncoder produce sparse vectors
# y StandardScaler con withMean=True requiere dense vectors (lanzaría error).
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

print("Pipeline de preprocesamiento definido ✓")
print(f"Features numéricas ({len(numeric_cols)}): {numeric_cols}")
print(f"Features categóricas (3): gender, region, primary_device")

In [ ]:
# ─── Modelo de Regresión Logística ─────────────────────────
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.5  # Mix L1/L2 regularization
)

# Pipeline completo: preprocesamiento → modelo
pipeline_lr = Pipeline(stages=[
    indexer_gender, indexer_region, indexer_device,
    encoder_gender, encoder_region, encoder_device,
    assembler, scaler, lr
])

print("Pipeline de Regresión Logística construido ✓")
print(f"Total de stages: {len(pipeline_lr.getStages())}")
for i, stage in enumerate(pipeline_lr.getStages()):
    print(f"  [{i}] {type(stage).__name__}")

In [ ]:
# ─── Train/Test Split y Entrenamiento ──────────────────────
start = time.time()

# Split 80/20
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count = test_df.count()
print(f"Train: {train_count:,} | Test: {test_count:,}")

# Verificar distribución de label en ambos sets
print("\nDistribución de label en train:")
train_df.groupBy("label").count().show()
print("Distribución de label en test:")
test_df.groupBy("label").count().show()

# Entrenar
model_lr = pipeline_lr.fit(train_df)
elapsed = time.time() - start
print(f"Modelo entrenado en {elapsed:.2f} seg ✓")

In [ ]:
# ─── Evaluación del Modelo Supervisado ─────────────────────
# Predicciones sobre test set
predictions_lr = model_lr.transform(test_df)

print("Muestra de predicciones:")
predictions_lr.select("user_id", "total_spent", "label", "prediction", "probability")     .show(10, truncate=False)

# Métricas de evaluación
evaluator_bin = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC"
)
evaluator_multi = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction"
)

auc = evaluator_bin.evaluate(predictions_lr)
accuracy = evaluator_multi.evaluate(predictions_lr, {evaluator_multi.metricName: "accuracy"})
f1 = evaluator_multi.evaluate(predictions_lr, {evaluator_multi.metricName: "f1"})
precision = evaluator_multi.evaluate(predictions_lr, {evaluator_multi.metricName: "weightedPrecision"})
recall = evaluator_multi.evaluate(predictions_lr, {evaluator_multi.metricName: "weightedRecall"})

print("\n" + "=" * 50)
print("  MÉTRICAS — REGRESIÓN LOGÍSTICA")
print("=" * 50)
print(f"  AUC-ROC:    {auc:.4f}")
print(f"  Accuracy:   {accuracy:.4f}")
print(f"  F1-Score:   {f1:.4f}")
print(f"  Precision:  {precision:.4f}")
print(f"  Recall:     {recall:.4f}")
print("=" * 50)

In [ ]:
# ─── Matriz de Confusión ───────────────────────────────────
print("📊 MATRIZ DE CONFUSIÓN")
print("(label = real, prediction = predicción del modelo)")
predictions_lr.groupBy("label", "prediction").count()     .orderBy("label", "prediction").show()

# Coeficientes del modelo (última stage del pipeline)
lr_model = model_lr.stages[-1]  # El modelo de LogisticRegression entrenado

print("Coeficientes del modelo (features numéricas):")
coefficients = lr_model.coefficients.toArray()
for i, feat in enumerate(numeric_cols):
    if i < len(coefficients):
        print(f"  {feat:25s}: {coefficients[i]:+.4f}")

print(f"\nIntercept: {lr_model.intercept:.4f}")

### 5.4 Modelo No Supervisado: K-Means (Segmentación)

**Objetivo:** Segmentar usuarios en grupos basándose en su comportamiento de compra, sin usar etiquetas predefinidas. Esto permite descubrir **segmentos naturales** en los datos.

In [ ]:
# ─── Preparación de features para K-Means ─────────────────
# Usar solo features numéricas de comportamiento (sin categóricas)
kmeans_features = [
    "total_purchases", "avg_ticket", "std_ticket",
    "avg_quantity", "avg_discount", "n_categories",
    "n_products", "n_payment_methods", "age"
]

assembler_km = VectorAssembler(
    inputCols=kmeans_features,
    outputCol="features_km_raw",
    handleInvalid="skip"
)

# MinMaxScaler es mejor para K-Means (rango [0,1] uniforme)
scaler_km = MinMaxScaler(
    inputCol="features_km_raw",
    outputCol="features_km"
)

# Aplicar transformaciones
pipeline_prep_km = Pipeline(stages=[assembler_km, scaler_km])
model_prep_km = pipeline_prep_km.fit(df_user_features)
df_km = model_prep_km.transform(df_user_features)
df_km.cache()

print(f"Features K-Means preparadas: {df_km.count():,} usuarios")
print(f"Features: {kmeans_features}")

In [ ]:
# ─── Búsqueda del K óptimo (Silhouette Score) ─────────────
print("🔍 Evaluando K de 2 a 8...")
print(f"{'K':>3} | {'Silhouette':>12} | {'Costo (WSSSE)':>15}")
print("-" * 40)

evaluator_km = ClusteringEvaluator(
    featuresCol="features_km",
    metricName="silhouette"
)

results_k = []
for k in range(2, 9):
    kmeans = KMeans(
        k=k, seed=42,
        featuresCol="features_km",
        predictionCol="cluster_temp",
        maxIter=50
    )
    model_km_temp = kmeans.fit(df_km)
    predictions_temp = model_km_temp.transform(df_km)

    silhouette = evaluator_km.evaluate(predictions_temp)
    cost = model_km_temp.summary.trainingCost

    results_k.append({'k': k, 'silhouette': silhouette, 'cost': cost})
    print(f"{k:3d} | {silhouette:12.4f} | {cost:15.2f}")

    # Limpiar columna temporal para siguiente iteración
    df_km = df_km.drop("cluster_temp")

# Seleccionar mejor K
best_k = max(results_k, key=lambda x: x['silhouette'])
print(f"\n✓ Mejor K = {best_k['k']} (Silhouette = {best_k['silhouette']:.4f})")

In [ ]:
# ─── Entrenar modelo final con K óptimo ────────────────────
K_FINAL = best_k['k']

kmeans_final = KMeans(
    k=K_FINAL, seed=42,
    featuresCol="features_km",
    predictionCol="cluster",
    maxIter=100
)

start = time.time()
model_km_final = kmeans_final.fit(df_km)
elapsed = time.time() - start

df_clustered = model_km_final.transform(df_km)
print(f"K-Means (K={K_FINAL}) entrenado en {elapsed:.2f} seg ✓")
print(f"Costo final (WSSSE): {model_km_final.summary.trainingCost:.2f}")

In [ ]:
# ─── Análisis de Clusters ──────────────────────────────────
print("=" * 70)
print(f"  PERFIL DE CLUSTERS (K={K_FINAL})")
print("=" * 70)

# Distribución de usuarios por cluster
print("\n📊 Distribución:")
df_clustered.groupBy("cluster").count().orderBy("cluster").show()

# Perfil promedio de cada cluster
print("📊 Perfil promedio por cluster:")
df_profiles = df_clustered.groupBy("cluster").agg(
    F.count("user_id").alias("n_usuarios"),
    F.round(F.avg("total_purchases"), 1).alias("prom_compras"),
    F.round(F.avg("avg_ticket"), 2).alias("prom_ticket"),
    F.round(F.avg("total_spent"), 2).alias("prom_gasto_total"),
    F.round(F.avg("avg_discount"), 1).alias("prom_descuento"),
    F.round(F.avg("n_categories"), 1).alias("prom_categorias"),
    F.round(F.avg("age"), 1).alias("prom_edad"),
).orderBy("cluster")

df_profiles.show(truncate=False)

In [ ]:
# ─── Interpretación de Segmentos para Marketing ────────────
print("=" * 70)
print("  INSIGHTS PARA MARKETING")
print("=" * 70)

# Cruzar clusters con segmentos originales
print("\n📊 Clusters vs Segmentos originales:")
df_clustered.groupBy("cluster", "segment")     .count()     .orderBy("cluster", F.desc("count"))     .show(30)

# Cruzar con región
print("📊 Clusters vs Región:")
df_clustered.groupBy("cluster", "region")     .count()     .orderBy("cluster", F.desc("count"))     .show(30)

# Cruzar con género
print("📊 Clusters vs Género:")
df_clustered.groupBy("cluster", "gender")     .count()     .orderBy("cluster", F.desc("count"))     .show(20)

In [ ]:
# ─── Nombrar los clusters según su perfil ──────────────────
profiles = df_profiles.collect()

print("\n🎯 SEGMENTOS IDENTIFICADOS (PROPUESTA):")
print("=" * 70)
for row in profiles:
    cluster_id = row['cluster']
    n = row['n_usuarios']
    ticket = row['prom_ticket']
    compras = row['prom_compras']
    gasto = row['prom_gasto_total']
    edad = row['prom_edad']

    # Lógica de naming basada en comportamiento
    if gasto > 2000 and compras > 15:
        nombre = "VIP / Alta Frecuencia"
        accion = "Programa de fidelización premium, acceso anticipado a ofertas"
    elif ticket > 200 and compras < 10:
        nombre = "Alto Ticket / Baja Frecuencia"
        accion = "Campañas de reactivación, cross-selling personalizado"
    elif compras > 10 and ticket < 100:
        nombre = "Comprador Frecuente / Bajo Ticket"
        accion = "Upselling, bundles de productos, descuentos por volumen"
    elif gasto < 500:
        nombre = "Nuevos / Exploradores"
        accion = "Onboarding, cupones de primera compra, educación de producto"
    else:
        nombre = "Intermedio / Estable"
        accion = "Mantener engagement, newsletters personalizadas"

    print(f"\nCluster {cluster_id}: {nombre}")
    print(f"  Usuarios: {n:,} | Gasto prom: ${gasto:,.0f} | Ticket: ${ticket:,.0f} | Compras: {compras:.0f}")
    print(f"  Edad prom: {edad:.0f} años")
    print(f"  Acción recomendada: {accion}")

print("\n" + "=" * 70)

In [ ]:
# ─── Guardar resultados ────────────────────────────────────
os.makedirs(PARQUET_DIR, exist_ok=True)

# Guardar usuarios con su cluster asignado
df_clustered.select(
    "user_id", "total_purchases", "total_spent", "avg_ticket",
    "n_categories", "age", "gender", "region", "segment", "cluster"
).write.mode("overwrite").parquet(
    os.path.join(PARQUET_DIR, "users_clustered")
)
print("✓ users_clustered.parquet guardado")

# Guardar predicciones del modelo supervisado
predictions_lr.select(
    "user_id", "label", "prediction", "probability"
).write.mode("overwrite").parquet(
    os.path.join(PARQUET_DIR, "predictions_high_value")
)
print("✓ predictions_high_value.parquet guardado")

In [ ]:
# ─── RESUMEN FINAL DEL PROYECTO ────────────────────────────
print("\n" + "=" * 70)
print("  RESUMEN FINAL — RETAIL ANALYTICS PIPELINE")
print("=" * 70)
print()
print("  L1: Fundamentos de Big Data")
print("     5V aplicadas a RetailMax, arquitectura definida")
print()
print("  L2: Configuración de Spark")
print("     SparkSession + SparkContext operativos, 951K registros cargados")
print()
print("  L3: RDDs, Transformaciones y Acciones")
print("     map, filter, flatMap, distinct, sortBy, reduceByKey")
print("     count, collect, reduce, mean, stdev + DAG documentado")
print()
print("  L4: Spark SQL y DataFrames")
print("     DataFrames con StructType, 6+ consultas SQL, Parquet output")
print()
print("  L5: MLlib Pipeline")
print(f"     Regresión Logística: AUC={auc:.4f}, F1={f1:.4f}")
print(f"     K-Means: K={K_FINAL}, Silhouette={best_k['silhouette']:.4f}")
print("     Segmentación accionable para marketing")
print()
print("=" * 70)
print("  Pipeline completo — Listo para entrega")
print("=" * 70)

In [ ]:
# ─── Cerrar Spark ──────────────────────────────────────────
spark.stop()
print("SparkSession cerrada ✓")